# Prediksi Risiko Eutrofikasi Danau Menggunakan Data Science

**Nama:** Septian Dwi Saputra  
**NIM:** 411232056  
**Mata Kuliah:** Analitik dan Visualisasi Data

> **Penting:** dataset pada proyek ini adalah data simulasi untuk pembelajaran, bukan hasil pengukuran lapangan. Nilai evaluasi hanya berlaku pada simulasi dan tidak menyatakan performa untuk danau nyata.

## 1. Persiapan dan pembentukan dataset

Skrip sumber `analisis_eutrofikasi.py` berisi pembangkit dataset dengan random seed tetap, perhitungan Carlson TSI, pelatihan dua model, evaluasi berbasis kelompok, dan pembuatan grafik. Jalankan sel berikut untuk menghasilkan ulang semua keluaran. Validasi silang untuk memilih model hanya memakai data latih; kelompok danau pada data uji tidak ikut menentukan pemilihan model.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image

from analisis_eutrofikasi import generate_dataset, run_analysis, FEATURES, ROOT, FIG_DIR, OUT_DIR

df = generate_dataset()
metrics = run_analysis(df)
print(f"Jumlah sampel: {len(df):,}")
print(f"Jumlah danau simulasi: {df['lake_id'].nunique()}")
print(f"Status dataset: {df['data_status'].iloc[0]}")

## 2. Memahami data dan kualitas data

Setiap baris merepresentasikan satu observasi simulasi. Data memuat delapan fitur prediktor, tiga indikator untuk menghitung TSI, serta label kelas hasil perhitungan TSI. Kode danau sengaja memakai awalan `SIM-LAKE` agar tidak dianggap sebagai nama danau nyata.

In [ ]:
display(df.head(10))
print("Ukuran dataset:", df.shape)
print("Jumlah nilai kosong:", int(df.isna().sum().sum()))
print("Jumlah baris duplikat:", int(df.duplicated().sum()))
display(df[FEATURES].describe().T.round(3))

## 3. Perhitungan Carlson Trophic State Index

Komponen TSI dihitung dari total fosfor (µg/L), klorofil-a (µg/L), dan kedalaman Secchi (m). Pada proyek ini rata-rata ketiga komponen dipakai sebagai CTSI komposit operasional; rata-rata tersebut adalah keputusan metodologis tugas ini, bukan satu-satunya cara baku penilaian danau. Kelas operasional proyek adalah oligotrofik jika CTSI < 40, mesotrofik jika 40 ≤ CTSI ≤ 50, dan eutrofik jika CTSI > 50. Batas tersebut bukan ambang universal atau regulasi untuk semua danau. Kelas status trofik dipakai sebagai proksi sederhana risiko eutrofikasi, bukan prediksi langsung kejadian HAB atau toksin.

In [ ]:
def calculate_tsi(tp_ug_l, chlorophyll_ug_l, secchi_m):
    tsi_tp = 14.42 * np.log(tp_ug_l) + 4.15
    tsi_chla = 9.81 * np.log(chlorophyll_ug_l) + 30.6
    tsi_secchi = 60 - 14.41 * np.log(secchi_m)
    return tsi_tp, tsi_chla, tsi_secchi

check = df.sample(5, random_state=61).copy()
check[["tsi_tp_check", "tsi_chla_check", "tsi_secchi_check"]] = check.apply(
    lambda r: pd.Series(calculate_tsi(r.total_phosphorus_ug_l,
                                      r.chlorophyll_a_ug_l,
                                      r.secchi_depth_m)), axis=1)
check["ctsi_check"] = check[["tsi_tp_check", "tsi_chla_check", "tsi_secchi_check"]].mean(axis=1)
display(check[["sample_id", "carlson_tsi_composite", "ctsi_check", "eutrophication_class"]].round(3))

## 4. Analisis deskriptif dan visualisasi

Distribusi kelas memperlihatkan ketidakseimbangan jumlah kelas. Hal tersebut perlu diperhatikan saat mengevaluasi model; karena itu Macro F1 dan balanced accuracy dilaporkan, bukan hanya accuracy.

In [ ]:
display(df["eutrophication_class"].value_counts().rename("jumlah").to_frame())
Image(filename=str(FIG_DIR / "01_distribusi_kelas.png"))

In [ ]:
Image(filename=str(FIG_DIR / "02_sebaran_ctsi.png"))

In [ ]:
Image(filename=str(FIG_DIR / "03_tp_chla_kelas.png"))

In [ ]:
Image(filename=str(FIG_DIR / "04_korelasi_fitur.png"))

## 5. Pemodelan tanpa target leakage

Fitur model adalah total nitrogen, suhu air, oksigen terlarut, curah hujan, persentase penggunaan lahan pertanian, persentase penggunaan lahan perkotaan, kedalaman rata-rata danau, dan pH. Total fosfor, klorofil-a, kedalaman Secchi, CTSI, serta komponennya **tidak** digunakan sebagai fitur karena variabel-variabel tersebut membentuk label target. Memasukkan variabel pembentuk label dapat membuat evaluasi terlihat terlalu bagus karena model diberi informasi yang sudah dipakai untuk menentukan jawaban.

In [ ]:
print("Fitur prediktor:")
for feature in FEATURES:
    print("-", feature)
print("\nVariabel yang dikeluarkan untuk menghindari target leakage:")
print("TP, klorofil-a, Secchi, seluruh komponen TSI, dan CTSI komposit")

## 6. Perbandingan model dan evaluasi

Pemisahan train/test dilakukan berdasarkan `lake_id` dengan `GroupShuffleSplit`, sehingga seluruh observasi dari satu danau simulasi hanya berada pada satu bagian. Validasi silang memakai `StratifiedGroupKFold` sebanyak lima fold **hanya pada data latih**. Data uji disimpan terpisah dan tidak ikut menentukan model terpilih. Pemilihan model dilakukan berdasarkan rata-rata Macro F1 grouped cross-validation; performa akhir dilaporkan pada kelompok danau uji.

In [ ]:
comparison = pd.read_csv(OUT_DIR / "perbandingan_model.csv")
display(comparison.round(4))
print("Model terpilih:", metrics["selected_model"])
print("Accuracy:", round(metrics["test_accuracy"], 4))
print("Balanced accuracy:", round(metrics["test_balanced_accuracy"], 4))
print("Macro F1:", round(metrics["test_macro_f1"], 4))
print("Macro ROC-AUC OVR:", round(metrics["test_roc_auc_ovr_macro"], 4))
print("Grouped CV Macro F1 mean:", round(metrics["cv_macro_f1_mean"], 4))

In [ ]:
Image(filename=str(FIG_DIR / "05_confusion_matrix.png"))

In [ ]:
Image(filename=str(FIG_DIR / "06_perbandingan_model.png"))

In [ ]:
Image(filename=str(FIG_DIR / "07_feature_importance.png"))
importance = pd.read_csv(OUT_DIR / "feature_importance_permutation.csv")
display(importance.sort_values("importance", ascending=False).round(4))

## 7. Contoh inferensi

Prediksi berikut hanya contoh pada satu baris dataset simulasi. Untuk penerapan nyata, nilai fitur harus berasal dari hasil pengukuran laboratorium/monitoring dan model harus dilatih serta divalidasi ulang dengan data danau yang sesuai.

In [ ]:
import joblib
bundle = joblib.load(OUT_DIR / "model_klasifikasi_eutrofikasi.joblib")
model = bundle["model"]
sample = df[FEATURES].iloc[[0]]
prediction = model.predict(sample)[0]
probability = model.predict_proba(sample)[0]
print("Prediksi kelas:", prediction)
print("Probabilitas per kelas:")
display(pd.DataFrame({"kelas": model.classes_, "probabilitas": probability}).sort_values("probabilitas", ascending=False))

## 8. Kesimpulan dan batasan

Eksperimen menunjukkan bahwa alur data science dapat digunakan untuk mengklasifikasikan status trofik dalam skenario simulasi. Hasil model tidak dapat digeneralisasi ke danau nyata karena tidak dilatih menggunakan pengukuran lapangan. Tahap berikutnya adalah mengganti dataset simulasi dengan data pemantauan aktual, menyelaraskan satuan dan waktu sampling, menetapkan ambang sesuai karakteristik wilayah, dan menguji model pada danau yang benar-benar terpisah dari data pelatihan.

**Sumber metodologi:** Carlson (1977), *A trophic state index for lakes*, DOI: https://doi.org/10.4319/lo.1977.22.2.0361; U.S. EPA, *Nutrient Criteria Technical Guidance Manual: Lakes and Reservoirs*; dataset terbuka EPA: https://doi.org/10.23719/1529835.